# B19 - Forward Diffusion on Fashion-MNIST (Linear schedule, 100 steps)
a. Corrupt images gradually over **100 steps** using a **linear** noise schedule.  b. Show noisy images at **t = 10 and t = 100**.

Forward process (closed form): $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$, with $\epsilon\sim\mathcal{N}(0,I)$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
(x_train, y_train), _ = keras.datasets.fashion_mnist.load_data()
x0 = x_train[:6].astype("float32") / 127.5 - 1.0      # scale to [-1, 1]

In [ ]:
# Linear noise schedule, T = 100 (betas scaled by 1000/T so x_T is close to pure noise)
def linear_schedule(T):
    betas = np.linspace(1e-4 * 1000 / T, 0.02 * 1000 / T, T)
    alpha_bar = np.concatenate([[1.0], np.cumprod(1 - betas)])
    return betas, alpha_bar

betas, alpha_bar = linear_schedule(100)

In [ ]:
class ForwardDiffusion:
    def __init__(self, alpha_bar):
        self.alpha_bar = tf.constant(alpha_bar, dtype=tf.float32)    # index t = 0..T

    def q_sample(self, x0, t):
        ab = tf.gather(self.alpha_bar, t)
        eps = tf.random.normal(tf.shape(x0))
        return tf.sqrt(ab) * x0 + tf.sqrt(1.0 - ab) * eps

T = 100
diffusion = ForwardDiffusion(alpha_bar)

plt.plot(alpha_bar); plt.xlabel('t'); plt.ylabel('alpha_bar_t'); plt.title('Noise schedule'); plt.show()

In [ ]:
# Gradual corruption of one image
steps = np.linspace(0, T, 9).astype(int)
plt.figure(figsize=(14, 2))
for i, t in enumerate(steps):
    plt.subplot(1, len(steps), i + 1)
    plt.imshow(diffusion.q_sample(x0[0], int(t)), cmap='gray'); plt.title(f't={t}'); plt.axis('off')
plt.suptitle('Gradual corruption'); plt.show()

In [ ]:
# Required steps: t = [10, 100]
show = [10, 100]
n = len(x0)
plt.figure(figsize=(2 * n, 2 * (len(show) + 1)))
for i in range(n):
    plt.subplot(len(show) + 1, n, i + 1); plt.imshow(x0[i], cmap='gray'); plt.axis('off')
    if i == 0: plt.title('original', loc='left')
    for r, t in enumerate(show):
        plt.subplot(len(show) + 1, n, (r + 1) * n + i + 1)
        plt.imshow(diffusion.q_sample(x0[i], t), cmap='gray'); plt.axis('off')
        if i == 0: plt.title(f't={t}', loc='left')
plt.show()

for t in show:
    print(f"t={t}: alpha_bar={alpha_bar[t]:.4f}  signal weight={np.sqrt(alpha_bar[t]):.4f}  noise weight={np.sqrt(1-alpha_bar[t]):.4f}")